#BIOETANOL SIMULATOR
Proyek ini bertujuan untuk merancang sistem simulasi flash drum yang bertujuan untuk memurnikan dan memisahkan dua atau lebih senyawa menjadi dua aliran (distillate dan bottom product) yang berbeda tingkat komposisi senyawanya. Perbedaan komposisi tersebut dipengaruhi oleh kondisi kesetimbangan fasa uap-cair dari senyawa umpan.
#OUTPUT PROYEK
Setelah proyek ini selesai, diharapkan simulasi mampu melakukan:
1. Penentuan laju alir total uap (V) dan cairan (L) yang terbentuk.
2. Penentuan komposisi fraksi mol senyawa di fase uap (y) dan di fase cair (x).
3. Penentuan bubble point dan dew point.

#FASE 1
Pembuatan database menggunakan library SQLite untuk menyimpan parameter konstanta Antoine (A, B, dan C) serta batas suhu bawah (Tmin) dan batas suhu atas (Tmax) persamaan tersebut valid digunakan.

In [45]:
import sqlite3

# 1. Membuat atau membuka koneksi ke file database lokal
# Jika file properties.db belum ada, Python akan otomatis membuatnya di folder proyek
conn = sqlite3.connect('properties.db')

# 2. Membuat objek cursor untuk mengeksekusi perintah SQL
cursor = conn.cursor()

# 3. Menulis query SQL untuk membuat tabel (LENGKAPI BAGIAN INI)
query_buat_tabel = '''
CREATE TABLE IF NOT EXISTS antoine_constants (
    id INTEGER PRIMARY KEY, 
    nama_komponen TEXT UNIQUE,    --Nama bahan kimia (misal: Etanol)
    A REAL,                     --Konstanta Antoine A
    B REAL,                     --Konstanta Antoine B
    C REAL,                     --Konstanta Antoine C
    T_min REAL,                 --Batas bawah suhu valid
    T_max REAL,                 --Batas atas suhu valid
    satuan_T TEXT,              --Satuan suhu yang digunakan (misal: Kelvin)
    satuan_P TEXT               --Satuan tekanan yang digunakan (misal: Bar)  
)
'''

# 4. Mengeksekusi query
cursor.execute(query_buat_tabel)

# 5. Menyimpan perubahan (commit) dan menutup koneksi
conn.commit()
conn.close()

print("Database properties.db dan tabel antoine_constants berhasil dibuat!")

Database properties.db dan tabel antoine_constants berhasil dibuat!


#FASE 1.B
Menginputkan data-data yang dibutuhkan ke dalam tabel yang telah dibuat.

In [46]:
import sqlite3

# Membuka kembali koneksi ke database
conn = sqlite3.connect('properties.db')
cursor = conn.cursor()

# Data konstanta dari Appendix B (Smith, Van Ness, Abbott)
# Format urutan di dalam tuple harus sama persis dengan urutan kolom di tabel
# (nama_komponen, A, B, C, T_min, T_max, satuan_T, satuan_P)
data_bahan = [
    ('Etanol', 16.8958, 3795.17, 230.918, 3.0, 96.0, 'Celcius', 'kPa'),
    ('Air', 16.3872, 3885.70, 230.170, 0.0, 200.0, 'Celcius', 'kPa')
    # Tulis komponen baru Anda di baris bawah ini:
    # ('Propanol', ... )
]

# Query INSERT menggunakan placeholder tanda tanya (?) untuk mencegah SQL Injection
query_insert = '''
INSERT OR IGNORE INTO antoine_constants (nama_komponen, A, B, C, T_min, T_max, satuan_T, satuan_P)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
'''

# executemany memungkinkan kita memasukkan puluhan/ratusan baris data sekaligus
cursor.executemany(query_insert, data_bahan)

# Wajib melakukan commit agar data benar-benar tersimpan ke file .db
conn.commit()
conn.close()

print("Data Komponen berhasil dimasukkan ke dalam tabel antoine_constants!")

Data Komponen berhasil dimasukkan ke dalam tabel antoine_constants!


In [47]:
import sqlite3

conn = sqlite3.connect('properties.db')
cursor = conn.cursor()

# Membuat tabel khusus interaksi biner (Composite Key untuk 2 komponen)
query_buat_tabel_biner = '''
CREATE TABLE IF NOT EXISTS van_laar_constants (
    komponen_1 TEXT,
    komponen_2 TEXT,
    A12 REAL,
    A21 REAL,
    PRIMARY KEY (komponen_1, komponen_2) 
)
'''
cursor.execute(query_buat_tabel_biner)

# Menyuntikkan data matriks interaksi
data_interaksi = [
    ('Etanol', 'Air', 1.6798, 0.9227),
    ('Metanol', 'Air', 0.8037, 0.5619),
    ('Metanol', 'Etanol', 0.0, 0.0)
    # Tulis matriks pasangan komponen baru Anda di bawah ini:
    # ('Propanol', 'Air', nilai_A12, nilai_A21)
]

# INSERT OR IGNORE mencegah error jika data sudah ada saat cell dijalankan berulang
cursor.executemany('''
INSERT OR IGNORE INTO van_laar_constants (komponen_1, komponen_2, A12, A21)
VALUES (?, ?, ?, ?)
''', data_interaksi)

conn.commit()
conn.close()

print("Tabel matriks interaksi biner berhasil dibuat di properties.db!")

Tabel matriks interaksi biner berhasil dibuat di properties.db!


#FASE 1.C
Hubungkan database dengan Python secara real time menggunakan pandas

In [48]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('properties.db')

# Menarik tabel Antoine
df_komponen = pd.read_sql_query("SELECT * FROM antoine_constants", conn)

# Menarik tabel Van Laar
df_interaksi = pd.read_sql_query("SELECT * FROM van_laar_constants", conn)

conn.close()

# Tampilkan tabel interaksi untuk validasi visual
df_interaksi

,komponen_1,komponen_2,A12,A21
0,Etanol,Air,1.6798,0.9227
1,Metanol,Air,0.8037,0.5619
2,Metanol,Etanol,0.0000,0.0000


#FASE 2
Membangun "Wadah" (OOP). Merancang agar variabel P, T, Feed, x, dan y bisa diubah-ubah dan disimpan dengan rapi di memori komputer.

In [49]:
class Stream:
    # Fungsi __init__ adalah konstruktor untuk membangun objek aliran baru
    def __init__(self, nama, T, P, flow_rate, komposisi):
        self.nama = nama
        self.T = T                  # Suhu operasi (Celcius)
        self.P = P                  # Tekanan operasi (kPa)
        self.flow_rate = flow_rate  # Total laju alir (kmol/jam)
        
        # Komposisi menggunakan struktur data Dictionary {'Komponen': fraksi_mol}
        self.komposisi = komposisi  
        
        # Validasi dasar Asas Teknik Kimia: Total fraksi mol harus = 1
        total_fraksi = sum(self.komposisi.values())
        if abs(total_fraksi - 1.0) > 1e-6: # Toleransi error kecil
            print(f"Peringatan: Total fraksi mol di stream '{self.nama}' tidak sama dengan 1!")

    # Fungsi/Metode untuk menampilkan status aliran
    def tampilkan_info(self):
        print(f"--- Data Aliran: {self.nama} ---")
        print(f"Suhu       : {self.T} °C")
        print(f"Tekanan    : {self.P} kPa")
        print(f"Laju Alir  : {self.flow_rate} kmol/h")
        print(f"Komposisi  : {self.komposisi}\n")

In [50]:
# 1. Membuat objek (instantiation) dan menyimpannya di dalam variabel
#nama_variabel = NamaClass("Nama Aliran", nilai_suhu, nilai_tekanan, nilai_laju_alir, {'Komponen1': fraksi1, 'Komponen2': fraksi2})

# 2. Memanggil fungsi/metode milik objek tersebut menggunakan tanda titik (.)
#nama_variabel.nama_fungsi()

feed_stream = Stream("Feed", 85.0, 101.325, 100.0, {'Etanol': 0.3, 'Air': 0.7})

feed_stream.tampilkan_info()

--- Data Aliran: Feed ---
Suhu       : 85.0 °C
Tekanan    : 101.325 kPa
Laju Alir  : 100.0 kmol/h
Komposisi  : {'Etanol': 0.3, 'Air': 0.7}



#FASE 3
Pembuatan kalkulator untuk mengkalkulasi kondisi yang berada di flash drum
Rumus yang digunakan:
1. Menghitung Konstanta Kesetimbangan Fasa (Ki): Psat/Ptotal
2. Rachford_rice
3. Laju uap: F*(V/F) {Fz=Vy+Lx}
4. Laju Cair: L=F-V
5. Fraksi mol cair: x=z/(1+(V/F)*(Ki-1))
6. Fraksi mol uap: y=Ki.x

In [51]:
import numpy as np

class FlashDrum:
    # 1. Update __init__ untuk menerima df_interaksi
    def __init__(self, feed_stream, df_database, df_interaksi):
        self.feed = feed_stream  
        self.db = df_database    
        self.db_interaksi = df_interaksi # Menangkap dataframe Van Laar
        
        self.vapor_stream = None
        self.liquid_stream = None
        
    def hitung_Psat(self):
        print(f"Menghitung Tekanan Uap Murni pada T = {self.feed.T} °C...")
        P_sat = {}
        for komponen in self.feed.komposisi.keys():
            data = self.db.loc[self.db['nama_komponen'] == komponen].iloc[0]
            A, B, C = data['A'], data['B'], data['C']
            Psat_val = np.exp(A - (B / (self.feed.T + C)))
            P_sat[komponen] = Psat_val
            print(f"Psat {komponen}: {Psat_val:.3f} {data['satuan_P']}")
        return P_sat
    
    # 2. Update algoritma pencarian matriks Van Laar via Pandas
    def hitung_gamma_van_laar(self, k1, k2, x1_val, x2_val):
        
        # Mencari konfigurasi A-B
        match1 = self.db_interaksi[(self.db_interaksi['komponen_1'] == k1) & 
                                   (self.db_interaksi['komponen_2'] == k2)]
        # Mencari konfigurasi B-A (jika urutan dibalik)
        match2 = self.db_interaksi[(self.db_interaksi['komponen_1'] == k2) & 
                                   (self.db_interaksi['komponen_2'] == k1)]
        
        if not match1.empty:
            A12 = match1.iloc[0]['A12']
            A21 = match1.iloc[0]['A21']
        elif not match2.empty:
            # Jika tabel berisi A-B tapi user mencari B-A, balik nilai A12 dan A21
            A12 = match2.iloc[0]['A21']
            A21 = match2.iloc[0]['A12']
        else:
            A12, A21 = 0.0, 0.0 # Asumsi ideal jika pasangan tidak ada di database
        
        x1 = max(x1_val, 1e-10)
        x2 = max(x2_val, 1e-10)
        
        g1 = np.exp(A12 / (1 + (A12 * x1) / (A21 * x2))**2) if A12 != 0 else 1.0
        g2 = np.exp(A21 / (1 + (A21 * x2) / (A12 * x1))**2) if A21 != 0 else 1.0
        
        return g1, g2
        
    def selesaikan_neraca(self, ideal=True):
        P_sat = self.hitung_Psat()
        
        # Mengekstrak nama komponen secara otomatis dari Feed
        daftar_komp = list(self.feed.komposisi.keys())
        k1 = daftar_komp[0]
        k2 = daftar_komp[1]
        
        # Inisialisasi tebakan awal komposisi cair secara dinamis
        x_aktual = {k: self.feed.komposisi.get(k, 0.0) for k in daftar_komp}
        
        for iterasi_outer in range(50):
            
            # Kalkulasi Koefisien Aktivitas (Gamma)
            if ideal:
                gamma = {k1: 1.0, k2: 1.0}
            else:
                g1, g2 = self.hitung_gamma_van_laar(k1, k2, x_aktual[k1], x_aktual[k2])
                gamma = {k1: g1, k2: g2}
                
            K = {}
            for komp in daftar_komp:
                K[komp] = (gamma[komp] * P_sat[komp]) / self.feed.P
                
            def rachford_rice(VF):
                sigma = 0
                for komp, z_i in self.feed.komposisi.items():
                    sigma += (z_i * (K[komp] - 1)) / (1 + VF * (K[komp] - 1))
                return sigma
                
            batas_bawah = 0.0
            batas_atas = 1.0
            
            if rachford_rice(batas_bawah) < 0 or rachford_rice(batas_atas) > 0:
                print(f"\n[Mode Ideal={ideal}] PERINGATAN: Di luar kesetimbangan dua fasa.")
                return None
                
            for iterasi_inner in range(100):
                VF_tebakan = (batas_bawah + batas_atas) / 2
                error_rr = rachford_rice(VF_tebakan)
                
                if abs(error_rr) < 1e-6:
                    break
                elif rachford_rice(batas_bawah) * error_rr < 0:
                    batas_atas = VF_tebakan
                else:
                    batas_bawah = VF_tebakan
                    
            VF_final = (batas_bawah + batas_atas) / 2
            
            x_baru = {}
            for komp, z_i in self.feed.komposisi.items():
                x_baru[komp] = z_i / (1 + VF_final * (K[komp] - 1))
                
            # Evaluasi error berbasis komponen pertama secara dinamis
            error_x = abs(x_baru[k1] - x_aktual[k1])
            x_aktual = x_baru  
            
            if ideal or error_x < 1e-5:
                break 
                
        # --- PERHITUNGAN FINAL ---
        V = VF_final * self.feed.flow_rate
        L = self.feed.flow_rate - V
        y = {k: K[k] * v for k, v in x_aktual.items()}
        
        # Cetak Hasil
        tipe = "IDEAL (HUKUM RAOULT)" if ideal else "NON-IDEAL (VAN LAAR)"
        print(f"\n--- HASIL SIMULASI: {tipe} ---")
        print(f"Fraksi Uap (V/F)   : {VF_final:.4f}")
        print(f"Laju Uap (V)       : {V:.2f} kmol/h")
        print(f"Laju Cair (L)      : {L:.2f} kmol/h")
        
        # Mencetak Gamma dan Komposisi secara dinamis
        if not ideal:
            print(f"\nKoefisien Aktivitas (Gamma):")
            for k, v in gamma.items(): print(f" - {k}: {v:.4f}")
            
        print("\nKomposisi Cairan (x):")
        for k, v in x_aktual.items(): print(f" - {k}: {v:.4f}")
        print("Komposisi Uap (y):")
        for k, v in y.items(): print(f" - {k}: {v:.4f}")
        print("-" * 35)

#MENAMBAHKAN KOMPONEN BARU

In [52]:
import sqlite3

conn = sqlite3.connect('properties.db')
cursor = conn.cursor()

# Data Konstanta Antoine Metanol (Contoh dari literatur: Celcius, kPa) (BISA DATA LAINNYA)
data_baru = [
    ('Metanol', 16.5785, 3638.27, 239.500, -16.0, 91.0, 'Celcius', 'kPa')
]

query = '''
INSERT OR IGNORE INTO antoine_constants (nama_komponen, A, B, C, T_min, T_max, satuan_T, satuan_P)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
'''

cursor.executemany(query, data_baru)
conn.commit()
conn.close()

print("Data Metanol berhasil ditambahkan ke database!")

Data Metanol berhasil ditambahkan ke database!


#CELL SIMULASI

In [53]:
# Membuat aliran feed
feed_simulasi = Stream("Feed Simulasi", 92.0, 101.325, 100.0, {'Etanol': 0.3, 'Air': 0.7})

# Memasukkan aliran, database komponen murni, dan database interaksi ke Flash Drum
unit_flash_simulasi = FlashDrum(feed_simulasi, df_komponen, df_interaksi)

unit_flash_simulasi.selesaikan_neraca(ideal=True)
unit_flash_simulasi.selesaikan_neraca(ideal=False)

Menghitung Tekanan Uap Murni pada T = 92.0 °C...
Psat Etanol: 171.240 kPa
Psat Air: 75.654 kPa

--- HASIL SIMULASI: IDEAL (HUKUM RAOULT) ---
Fraksi Uap (V/F)   : 0.1696
Laju Uap (V)       : 16.96 kmol/h
Laju Cair (L)      : 83.04 kmol/h

Komposisi Cairan (x):
 - Etanol: 0.2686
 - Air: 0.7314
Komposisi Uap (y):
 - Etanol: 0.4539
 - Air: 0.5461
-----------------------------------
Menghitung Tekanan Uap Murni pada T = 92.0 °C...
Psat Etanol: 171.240 kPa
Psat Air: 75.654 kPa

[Mode Ideal=False] PERINGATAN: Di luar kesetimbangan dua fasa.
